# YOLO11 Training - Road Defect & Pothole Detection

This notebook demonstrates how to validate the dataset, configure hyperparameters, train a YOLO11 model, evaluate it, plot performance curves, and export the model for production.

## Notebook Workflow:
1. **Environment Setup & .env Configuration**
2. **Dataset Validation & Sanity Checks**
3. **Hyperparameter Configuration**
4. **YOLO11 Training**
5. **Validation & Metrics Evaluation**
6. **Test-Set Evaluation**
7. **Confusion Matrix & Curves Generation**
8. **Model Export (ONNX)**
9. **Performance Benchmarking (FPS, Memory, Size)**
10. **Save Final Deliverables to `final_model/`**

## 1. Setup & Configuration

Verify the virtual environment, load variables from `.env`, and import libraries.

In [ ]:
import sys
import os
import shutil
import json
import time
import logging
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ═══════════════════════════════════════════════════════════════════════════════
# VERIFY ENVIRONMENT
# ═══════════════════════════════════════════════════════════════════════════════
python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
python_path = sys.executable
print(f"Using Python: {python_version} from {python_path}")

# Set project root path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'backend'))

try:
    from config import config
    print("✓ Backend config loaded successfully.")
    print(f"  ACTIVE_MODEL: {config.ACTIVE_MODEL}")
    print(f"  DEVICE: {config.DEVICE}")
except ImportError:
    print("⚠️ Could not load backend config directly. Will use manual fallback settings.")

In [ ]:
# Import Ultralytics & verify GPU availability
try:
    import torch
    from ultralytics import YOLO
    print("✓ PyTorch version:", torch.__version__)
    print("✓ CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("✓ GPU Device Name:", torch.cuda.get_device_name(0))
    print("✓ Ultralytics YOLO imported successfully.")
except ImportError as e:
    print("Installing dependencies...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "pandas", "matplotlib"])
    import torch
    from ultralytics import YOLO
    print("✓ Setup complete!")

## 2. Dataset Validation & Sanity Checks

Verify the dataset files, count annotations, and check class distributions.

In [ ]:
data_yaml_path = project_root / 'data' / 'data.yaml'
print(f"Checking dataset config: {data_yaml_path}")

if data_yaml_path.exists():
    with open(data_yaml_path, 'r') as f:
        yaml_content = f.read()
    print("\n--- data.yaml Content ---")
    print(yaml_content)
    print("-------------------------")
else:
    print("❌ data.yaml not found!")

# Validate split directories dynamically from data.yaml path
import yaml
with open(data_yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

dataset_dir = Path(data_cfg['path'])
train_images = dataset_dir / 'images' / 'train'
val_images = dataset_dir / 'images' / 'val'

if train_images.exists():
    train_count = len(list(train_images.glob('*')))
    val_count = len(list(val_images.glob('*')))
    print(f"\n✓ Dataset folders validated:")
    print(f"  - Path: {dataset_dir}")
    print(f"  - Training images: {train_count}")
    print(f"  - Validation images: {val_count}")
else:
    print(f"❌ Dataset directory not found at: {dataset_dir}")

## 3. Hyperparameter Configuration

Define the parameters we will pass to the YOLO11 model training pipeline.

In [ ]:
model_variant = "yolo11s" # Options: yolo11n, yolo11s, yolo11m
epochs = 50
batch_size = 8
img_size = 640
device = 0 if torch.cuda.is_available() else "cpu"
patience = 15

print("Training Hyperparameters:")
print(f"  - Model: {model_variant}.pt")
print(f"  - Epochs: {epochs}")
print(f"  - Batch Size: {batch_size}")
print(f"  - Device: {device}")
print(f"  - Patience: {patience}")

## 4. YOLO11 Training

Load pretrained weights and begin training on your custom dataset.

In [ ]:
print(f"Loading pretrained {model_variant} model...")
model = YOLO(f"{model_variant}.pt")

print("Starting YOLO11 training run...")
start_time = time.time()
results = model.train(
    data=str(data_yaml_path),
    epochs=epochs,
    imgsz=img_size,
    batch=batch_size,
    device=device,
    patience=patience,
    save=True,
    project=str(project_root / 'runs' / 'base_models'),
    name=f'pothole_detector_{model_variant}',
    workers=2,
    close_mosaic=5,
    plots=True
)
training_duration = time.time() - start_time
print(f"✓ Training completed in {training_duration:.2f} seconds ({training_duration/3600:.2f} hours).")

## 5. Validation & Metrics Evaluation

Validate the model on the verification split.

In [ ]:
print("Running validation split evaluation...")
val_metrics = model.val()

print("Validation Metrics:")
print(f"  - mAP50:       {val_metrics.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  - mAP50-95:    {val_metrics.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"  - Precision:   {val_metrics.results_dict['metrics/precision(B)']:.4f}")
print(f"  - Recall:      {val_metrics.results_dict['metrics/recall(B)']:.4f}")

## 6. Test-Set Evaluation

Evaluate the model by running inference on a batch of test/validation images.

In [ ]:
test_images_dir = val_images
print(f"Running batch inference on sample validation/test images inside: {test_images_dir}")

test_images = list(test_images_dir.glob('*.jpeg')) + list(test_images_dir.glob('*.jpg'))
if test_images:
    sample_test = test_images[:5]
    for img_path in sample_test:
        preds = model.predict(source=str(img_path), conf=0.25, save=False)
        print(f"  - {img_path.name}: Detected {len(preds[0].boxes)} defects")
else:
    print("⚠️ No test images found.")

## 7. Metrics & Confusion Matrix Plots

Find files containing confusion matrices, PR curves, and plot them here.

In [ ]:
# Find latest training run directory
base_models_dir = project_root / 'runs' / 'base_models'
run_dirs = sorted([d for d in base_models_dir.glob(f'pothole_detector_{model_variant}*') if d.is_dir()],
                  key=os.path.getmtime, reverse=True)

if run_dirs:
    latest_run_dir = run_dirs[0]
    print(f"✓ Found latest training output run: {latest_run_dir}")
    
    # Display confusion matrix
    conf_matrix_path = latest_run_dir / 'confusion_matrix.png'
    if conf_matrix_path.exists():
        print(f"Showing Confusion Matrix: {conf_matrix_path.name}")
        img = plt.imread(str(conf_matrix_path))
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
    
    # Display Precision-Recall Curve
    pr_curve_path = latest_run_dir / 'BoxPR_curve.png'
    if pr_curve_path.exists():
        print(f"Showing Precision-Recall Curve: {pr_curve_path.name}")
        img = plt.imread(str(pr_curve_path))
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("❌ Run outputs not found.")

## 8. Model Export (ONNX Format)

Export your trained PyTorch model `.pt` file to `ONNX` format.

In [ ]:
print("Exporting model to ONNX format...")
try:
    onnx_path = model.export(format='onnx')
    print(f"✓ Export complete: {onnx_path}")
except Exception as e:
    print(f"❌ Export failed: {e}")

## 9. Performance Benchmarking

Benchmark your trained model's inference speed, FPS, memory usage, and footprint.

In [ ]:
import psutil
print("Benchmarking inference latency and memory footprints...")

# Model size in MB
best_pt = latest_run_dir / 'weights' / 'best.pt'
model_size_mb = best_pt.stat().st_size / (1024 * 1024) if best_pt.exists() else 0.0

# Benchmarking inference speed
inference_times = []
dummy_image = np.zeros((640, 640, 3), dtype=np.uint8)

# Warmup runs
for _ in range(10):
    _ = model.predict(source=dummy_image, verbose=False)

# Measure speed
for _ in range(50):
    t_start = time.time()
    _ = model.predict(source=dummy_image, verbose=False)
    inference_times.append((time.time() - t_start) * 1000)

mean_inference_ms = np.mean(inference_times)
fps = 1000 / mean_inference_ms
cpu_memory_usage_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

print(f"Benchmark Results:")
print(f"  - Inference Time: {mean_inference_ms:.2f} ms per image")
print(f"  - FPS:            {fps:.1f} frames per second")
print(f"  - Model Size:     {model_size_mb:.2f} MB")
print(f"  - CPU Memory:     {cpu_memory_usage_mb:.2f} MB")

## 10. Save Final Deliverables to `final_model/` folder

Copy all output files, weights, and plots to the structured `final_model/` directory.

In [ ]:
final_model_dir = project_root / 'model_training' / 'final_model'
final_model_dir.mkdir(exist_ok=True)
print(f"Saving final deliverables to: {final_model_dir}")

# 1. Weights best.pt
if best_pt.exists():
    shutil.copy(best_pt, final_model_dir / 'yolo11_pothole.pt')

# 2. results.csv
results_csv = latest_run_dir / 'results.csv'
if results_csv.exists():
    shutil.copy(results_csv, final_model_dir / 'training_metrics.csv')

# 3. Confusion Matrix and Curves
if conf_matrix_path.exists():
    shutil.copy(conf_matrix_path, final_model_dir / 'confusion_matrix.png')
if pr_curve_path.exists():
    shutil.copy(pr_curve_path, final_model_dir / 'pr_curve.png')
f1_curve_path = latest_run_dir / 'BoxF1_curve.png'
if f1_curve_path.exists():
    shutil.copy(f1_curve_path, final_model_dir / 'f1_curve.png')

# 4. Evaluation Report JSON
eval_report = {
    "model_name": f"YOLO11s-Detailed-Cracks",
    "val_metrics": {
        "mAP50": float(val_metrics.results_dict.get('metrics/mAP50(B)', 0)),
        "mAP50_95": float(val_metrics.results_dict.get('metrics/mAP50-95(B)', 0)),
        "precision": float(val_metrics.results_dict.get('metrics/precision(B)', 0)),
        "recall": float(val_metrics.results_dict.get('metrics/recall(B)', 0))
    },
    "benchmark": {
        "inference_time_ms": float(mean_inference_ms),
        "fps": float(fps),
        "model_size_mb": float(model_size_mb),
        "cpu_memory_mb": float(cpu_memory_usage_mb)
    }
}

with open(final_model_dir / 'evaluation_report.json', 'w') as f:
    json.dump(eval_report, f, indent=4)

print("\n✓ All final deliverables copied and saved to final_model/")
print(list(final_model_dir.glob('*')))

---
# SECTION 2: Fine-Tuning on Pre-Cleaned Dataset

This section fine-tunes the trained YOLO11s model using the **pre-cleaned dataset** located in `dataset/clean_dataset/`.
Data cleaning, validation, and conversion from segmentations to bounding boxes have already been completed by the standalone `data_cleaning.py` script.

## 12. Fine-Tuning Hyperparameters

Configure parameters for fine-tuning. Key differences from training from scratch:
- **Lower learning rate** (0.001 vs 0.01) to preserve learned features
- **Shorter warmup** (3 epochs) since model is already trained
- **Cosine LR scheduler** for smooth convergence

In [ ]:
# Fine-tuning configuration
ft_checkpoint = project_root / 'runs' / 'base_models' / 'pothole_detector_yolo11s' / 'weights' / 'best.pt'
ft_epochs = 50
ft_batch_size = 8
ft_lr0 = 0.001        # Lower LR for fine-tuning
ft_lrf = 0.01         # Final LR = lr0 * lrf
ft_patience = 15
ft_output_name = 'pothole_detector_yolo11s_v2'
ft_device = 0 if torch.cuda.is_available() else 'cpu'

print("Fine-Tuning Configuration:")
print(f"  Checkpoint:   {ft_checkpoint}")
print(f"  Exists:       {ft_checkpoint.exists()}")
print(f"  Epochs:       {ft_epochs}")
print(f"  Batch Size:   {ft_batch_size}")
print(f"  Learning Rate: {ft_lr0}")
print(f"  Patience:     {ft_patience}")
print(f"  Device:       {ft_device}")
print(f"  Output:       {ft_output_name}")

## 13. Run Fine-Tuning

Load the existing trained model and fine-tune on the merged (expanded) dataset.

In [ ]:
print(f"Loading trained checkpoint: {ft_checkpoint}")
ft_model = YOLO(str(ft_checkpoint))
print("Model loaded successfully.")

print("\nStarting fine-tuning...")
ft_start = time.time()

ft_results = ft_model.train(
    data=str(data_yaml_path),
    epochs=ft_epochs,
    imgsz=640,
    batch=ft_batch_size,
    device=ft_device,
    patience=ft_patience,
    save=True,
    project=str(project_root / 'runs' / 'base_models'),
    name=ft_output_name,
    workers=2,
    close_mosaic=5,
    plots=True,
    # Fine-tuning specific
    lr0=ft_lr0,
    lrf=ft_lrf,
    warmup_epochs=3,
    cos_lr=True,
)

ft_duration = time.time() - ft_start
print(f"\nFine-tuning complete in {ft_duration:.0f}s ({ft_duration/3600:.2f} hours)")

## 14. Evaluate Fine-Tuned Model

Run validation on the fine-tuned model and compare with the original v1.

In [ ]:
print("Running validation on fine-tuned model...")
ft_val = ft_model.val()

print("\nFine-Tuned Model (v2) Metrics:")
print(f"  mAP@50:     {ft_val.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  mAP@50-95:  {ft_val.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"  Precision:  {ft_val.results_dict['metrics/precision(B)']:.4f}")
print(f"  Recall:     {ft_val.results_dict['metrics/recall(B)']:.4f}")

# Calculate F1
ft_p = ft_val.results_dict['metrics/precision(B)']
ft_r = ft_val.results_dict['metrics/recall(B)']
ft_f1 = 2 * ft_p * ft_r / (ft_p + ft_r + 1e-8)
print(f"  F1 Score:   {ft_f1:.4f}")

## 15. Compare v1 (Original) vs v2 (Fine-Tuned)

Side-by-side comparison of model performance before and after fine-tuning.

In [ ]:
# Load v1 metrics
v1_metrics_path = project_root / 'model_comparison_results' / 'exact_metrics.json'
if v1_metrics_path.exists():
    with open(v1_metrics_path) as f:
        v1_data = json.load(f)
    v1 = v1_data.get('YOLO11s', {})
else:
    print('Warning: exact_metrics.json not found, using zeros for v1')
    v1 = {'map50': 0, 'map50_95': 0, 'precision': 0, 'recall': 0}

v2 = {
    'map50': ft_val.results_dict['metrics/mAP50(B)'],
    'map50_95': ft_val.results_dict['metrics/mAP50-95(B)'],
    'precision': ft_val.results_dict['metrics/precision(B)'],
    'recall': ft_val.results_dict['metrics/recall(B)'],
}

v1_f1 = 2 * v1.get('precision',0) * v1.get('recall',0) / (v1.get('precision',0) + v1.get('recall',0) + 1e-8)
v2_f1 = 2 * v2['precision'] * v2['recall'] / (v2['precision'] + v2['recall'] + 1e-8)

print('=' * 70)
print('MODEL COMPARISON: v1 (Original) vs v2 (Fine-Tuned on Merged Data)')
print('=' * 70)
print(f'{"Metric":<15} {"v1 (Original)":>15} {"v2 (Fine-Tuned)":>17} {"Delta":>10}')
print('-' * 70)

metrics_compare = [
    ('mAP@50',    v1.get('map50', 0),    v2['map50']),
    ('mAP@50-95', v1.get('map50_95', 0), v2['map50_95']),
    ('Precision',  v1.get('precision', 0), v2['precision']),
    ('Recall',     v1.get('recall', 0),    v2['recall']),
    ('F1 Score',   v1_f1,                  v2_f1),
]

for name, val1, val2 in metrics_compare:
    delta = val2 - val1
    print(f'  {name:<13} {val1*100:>13.2f}% {val2*100:>15.2f}% {delta*100:>+9.2f}%')

print('=' * 70)

# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 6))
metric_names = [m[0] for m in metrics_compare]
v1_vals = [m[1] * 100 for m in metrics_compare]
v2_vals = [m[2] * 100 for m in metrics_compare]

x = np.arange(len(metric_names))
w = 0.32
bars1 = ax.bar(x - w/2, v1_vals, w, label='v1 (Original)', color='#2F80ED')
bars2 = ax.bar(x + w/2, v2_vals, w, label='v2 (Fine-Tuned)', color='#27AE60')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylabel('Percentage (%)')
ax.set_ylim(0, 100)
ax.set_title('YOLO11s v1 vs v2 (Fine-Tuned) Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(project_root / 'model_training' / 'report_assets' / 'v1_vs_v2_comparison.png', dpi=300)
plt.show()

## 16. Save Fine-Tuned Model & Update Deployment

If the fine-tuned model is better, save it and update the `.env` to deploy it.

In [ ]:
ft_run_dir = project_root / 'runs' / 'base_models' / ft_output_name
ft_best_pt = ft_run_dir / 'weights' / 'best.pt'

if ft_best_pt.exists():
    # Save to final_model
    ft_final_dir = project_root / 'model_training' / 'final_model'
    ft_final_dir.mkdir(exist_ok=True)
    
    shutil.copy(ft_best_pt, ft_final_dir / 'yolo11s_v2_finetuned.pt')
    print(f'Saved fine-tuned model to: {ft_final_dir / "yolo11s_v2_finetuned.pt"}')
    
    # Save v2 metrics JSON
    v2_report = {
        'model_name': 'YOLO11s-v2-FineTuned',
        'training': {
            'base_model': 'pothole_detector_yolo11s/weights/best.pt',
            'merged_datasets': ['02_DETAILED_CRACKS_ANNOTATION', '03_Pothole_Image_Segmentation_Datasets'],
            'epochs': ft_epochs,
            'lr0': ft_lr0,
        },
        'metrics': {
            'mAP50': float(v2['map50']),
            'mAP50_95': float(v2['map50_95']),
            'precision': float(v2['precision']),
            'recall': float(v2['recall']),
            'f1': float(v2_f1),
        },
    }
    
    with open(ft_final_dir / 'v2_evaluation_report.json', 'w') as f:
        json.dump(v2_report, f, indent=4)
    
    print(f'Saved v2 evaluation report.')
    
    # Check if v2 is better than v1
    if v2['map50'] > v1.get('map50', 0):
        print(f'\n>>> v2 IMPROVED mAP@50: {v1.get("map50",0)*100:.2f}% -> {v2["map50"]*100:.2f}%')
        print(f'>>> To deploy, update .env:')
        print(f'    BEST_MODEL_PATH=./runs/base_models/{ft_output_name}/weights/best.pt')
    else:
        print(f'\n>>> v2 did NOT improve mAP@50. Consider more epochs or hyperparameter tuning.')
else:
    print(f'Fine-tuned model not found at: {ft_best_pt}')